# Semantic-search evaluation

This notebook owns evaluation only. It judges complete legal articles by canonical article ID. Chunk retrieval is aggregated to article level. Relevance comes from `relevance_mapping.json`: 2 = direct, 1 = partial, 0 = not relevant.

In [ ]:
import json
import math
import unicodedata
from pathlib import Path

import numpy as np
from sentence_transformers import SentenceTransformer

def find_project_root(start):
    for directory in (start, *start.parents):
        if (directory / 'data-pipeline').is_dir():
            return directory
    raise RuntimeError('Run this notebook inside the legal-rag project')

PROJECT_ROOT = find_project_root(Path.cwd())
PROCESSED_DIR = PROJECT_ROOT / 'data-pipeline' / 'data' / 'processed'
EMBEDDING_DIR = PROCESSED_DIR / 'embeddings'
EVALUATION_DIR = PROJECT_ROOT / 'data-pipeline' / 'evaluation' / 'data'
model = SentenceTransformer('BAAI/bge-m3')

def load_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

def canonical_id(article):
    return str(article.get('article_id') or f"{article.get('document_id')}-{article.get('article_number')}")

raw_articles = load_jsonl(PROCESSED_DIR / 'articles_normalized.jsonl')
articles = {}
for article in raw_articles:
    key = canonical_id(article)
    current = articles.get(key)
    if current is None or len(article.get('text', '')) > len(current.get('text', '')):
        articles[key] = article

chunk_records = load_jsonl(EMBEDDING_DIR / 'chunks_bge_m3_recursive_450_tokens_metadata.jsonl')
chunk_vectors = np.load(EMBEDDING_DIR / 'chunks_bge_m3_recursive_450_tokens.npy')
article_vectors_path = EMBEDDING_DIR / 'articles_bge_m3.npy'
article_vectors = np.load(article_vectors_path)
article_list = list(articles.values())
assert len(chunk_records) == len(chunk_vectors)
assert len(article_list) == len(article_vectors)
print('Articles:', len(article_list), 'Chunks:', len(chunk_records))


In [ ]:
mapping = json.loads((EVALUATION_DIR / 'relevance_mapping.json').read_text(encoding='utf-8'))
qrels = {item['query_id']: {article_id: 2 for article_id in item['relevant_articles']} for item in mapping}
queries = {item['query_id']: item['query'] for item in mapping}
available_ids = set(articles)
eligible_qrels = {query_id: labels for query_id, labels in qrels.items() if set(labels) & available_ids}
excluded_query_ids = sorted(set(qrels) - set(eligible_qrels))
print('Queries:', len(qrels), 'Eligible:', len(eligible_qrels), 'Excluded:', excluded_query_ids)

def normalize_query(query):
    query = unicodedata.normalize('NFC', query)
    return ' '.join(query.replace('\u200b', '').replace('\u200c', '').replace('\u200d', '').split())

def rank_article_level(query_vector):
    order = np.argsort(-(article_vectors @ query_vector))
    return [canonical_id(article_list[index]) for index in order[:10]]

def rank_chunk_level(query_vector):
    scores = chunk_vectors @ query_vector
    best = {}
    for index in np.argsort(-scores):
        article_id = str(chunk_records[index]['article_id'])
        best[article_id] = max(best.get(article_id, -1), float(scores[index]))
    return [article_id for article_id, _ in sorted(best.items(), key=lambda item: item[1], reverse=True)[:10]]

def ndcg(ranked, labels, k):
    def gain(rel): return (2 ** rel) - 1
    top = ranked[:k]
    dcg = sum(gain(labels.get(article_id, 0)) / math.log2(index + 2) for index, article_id in enumerate(top))
    ideal = sorted(labels.values(), reverse=True)[:k]
    idcg = sum(gain(rel) / math.log2(index + 2) for index, rel in enumerate(ideal))
    return dcg / idcg if idcg else 0.0

def score(ranked, labels):
    relevant = {article_id for article_id, rel in labels.items() if rel > 0}
    ranks = [index + 1 for index, article_id in enumerate(ranked) if article_id in relevant]
    first = ranks[0] if ranks else None
    return {'hit@3': int(first is not None and first <= 3), 'hit@5': int(first is not None and first <= 5), 'recall@5': len(set(ranked[:5]) & relevant) / max(len(relevant), 1), 'mrr@3': 1 / first if first and first <= 3 else 0.0, 'ndcg@3': ndcg(ranked, labels, 3), 'rank': first}


In [ ]:
results = {'article_level': [], 'chunk_level': []}
for query_id, labels in eligible_qrels.items():
    vector = model.encode([normalize_query(queries[query_id])], convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)[0]
    for level, ranked in [('article_level', rank_article_level(vector)), ('chunk_level', rank_chunk_level(vector))]:
        results[level].append({'query_id': query_id, 'expected': labels, 'top10': ranked, **score(ranked, labels)})

def summarize(level):
    rows = results[level]
    metrics = ('hit@3', 'hit@5', 'recall@5', 'mrr@3', 'ndcg@3')
    return {metric: round(sum(row[metric] for row in rows) / len(rows), 4) for metric in metrics}

report = {'article_level': summarize('article_level'), 'chunk_level': summarize('chunk_level'), 'results': results}
output = PROCESSED_DIR / 'reports' / 'semantic_search_evaluation.json'
output.write_text(json.dumps(report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(json.dumps({key: report[key] for key in ('article_level', 'chunk_level')}, ensure_ascii=False, indent=2))
print('Saved:', output)
